# NorthStar Urban Mobility - Python Data Processing
## Part 3: Pandas, NumPy, Feature Engineering, Analysis & Visualisation

This notebook uses Python for data cleaning, feature engineering, advanced analysis, and visualisation to uncover operational inefficiencies across the NorthStar dataset.

## Setup
Run this cell first to clone the data repository.

In [ ]:
import os
if not os.path.exists('northstar-coursework'):
    !git clone https://github.com/Erucard/northstar-coursework.git
os.chdir('/content/northstar-coursework/data/raw')
print('Ready:', sorted([f for f in os.listdir('.') if f.endswith('.csv')]))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')


# DATA LOADING


data_path = '.'

hubs = pd.read_csv(f'{data_path}/hubs.csv')
customers = pd.read_csv(f'{data_path}/customers.csv')
drivers = pd.read_csv(f'{data_path}/drivers.csv')
vehicles = pd.read_csv(f'{data_path}/vehicles.csv')
orders = pd.read_csv(f'{data_path}/orders.csv')
deliveries = pd.read_csv(f'{data_path}/deliveries.csv')
incidents = pd.read_csv(f'{data_path}/incidents.csv')
complaints = pd.read_csv(f'{data_path}/complaints.csv')
app_events = pd.read_csv(f'{data_path}/app_events.csv')

print(f'Loaded: {len(hubs)} hubs, {len(customers)} customers, {len(drivers)} drivers,')
print(f'  {len(vehicles)} vehicles, {len(orders)} orders, {len(deliveries)} deliveries,')
print(f'  {len(incidents)} incidents, {len(complaints)} complaints, {len(app_events)} app_events')

In [ ]:

# STEP 1: DATA CLEANING - Zone Standardisation


zone_map = {
    'north': 'North', 'NORTH': 'North',
    'south': 'South', 'SOUTH': 'South',
    'east': 'East', 'EAST': 'East',
    'west': 'West', 'WEST': 'West',
    'central': 'Central', 'CENTRAL': 'Central', 'Ctr': 'Central',
    'airport': 'Airport', 'AIRPORT': 'Airport',
    'riverside': 'Riverside', 'RiverSide': 'Riverside'
}

# Apply zone standardisation
customers['home_zone'] = customers['home_zone'].replace(zone_map)
drivers['base_zone'] = drivers['base_zone'].replace(zone_map)
vehicles['assigned_zone'] = vehicles['assigned_zone'].replace(zone_map)
orders['pickup_zone'] = orders['pickup_zone'].replace(zone_map)
orders['dropoff_zone'] = orders['dropoff_zone'].replace(zone_map)
app_events['zone_context'] = app_events['zone_context'].replace(zone_map)

# Parse dates
orders['order_created_at'] = pd.to_datetime(orders['order_created_at'])
deliveries['dispatch_time'] = pd.to_datetime(deliveries['dispatch_time'])
deliveries['delivery_completed_at'] = pd.to_datetime(deliveries['delivery_completed_at'])
complaints['created_at'] = pd.to_datetime(complaints['created_at'])
incidents['reported_at'] = pd.to_datetime(incidents['reported_at'])
app_events['event_timestamp'] = pd.to_datetime(app_events['event_timestamp'])

print('Zone values after cleaning:')
print(f"  customers.home_zone: {sorted(customers['home_zone'].dropna().unique())}")
print(f"  orders.pickup_zone: {sorted(orders['pickup_zone'].dropna().unique())}")

In [ ]:

# STEP 2: MISSING VALUE ANALYSIS AND IMPUTATION


print('=== Missing Values Summary ===')
for name, df in [('customers', customers), ('drivers', drivers), ('vehicles', vehicles),
                  ('orders', orders), ('deliveries', deliveries), ('incidents', incidents),
                  ('complaints', complaints), ('app_events', app_events)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f'\n{name}:')
        for col, cnt in missing.items():
            print(f'  {col}: {cnt} missing ({cnt/len(df)*100:.1f}%)')

# Imputation strategies
# loyalty_score: median imputation (20 missing / 650 = 3.1%)
customers['loyalty_score'].fillna(customers['loyalty_score'].median(), inplace=True)

# preferred_channel: mode imputation (13 missing)
customers['preferred_channel'].fillna(customers['preferred_channel'].mode()[0], inplace=True)

# training_score: median by zone (7 missing)
drivers['training_score'] = drivers.groupby('base_zone')['training_score'].transform(
    lambda x: x.fillna(x.median()))

# battery_health_pct: median (4 missing)
vehicles['battery_health_pct'].fillna(vehicles['battery_health_pct'].median(), inplace=True)

# booking_channel: mode (25 missing)
orders['booking_channel'].fillna(orders['booking_channel'].mode()[0], inplace=True)

# compensation_amount: 0 for missing (16 missing - likely no compensation given)
complaints['compensation_amount'].fillna(0, inplace=True)

print('\n=== After Imputation: All datasets have 0 critical nulls ===')

In [ ]:

# STEP 3: FEATURE ENGINEERING


# Delivery time features
deliveries['delivery_hours'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600

deliveries['is_negative_time'] = (deliveries['delivery_hours'] < 0).astype(int)
deliveries['is_failed'] = (deliveries['delivery_status'] == 'Failed').astype(int)
deliveries['is_delayed'] = (deliveries['delivery_status'] == 'Delayed').astype(int)
deliveries['cost_per_km'] = deliveries['fuel_or_charge_cost'] / deliveries['route_distance_km']

# Time-based features from orders
orders['order_hour'] = orders['order_created_at'].dt.hour
orders['order_dayofweek'] = orders['order_created_at'].dt.dayofweek
orders['is_weekend'] = (orders['order_dayofweek'] >= 5).astype(int)
orders['order_month'] = orders['order_created_at'].dt.month

# Customer features
complaint_counts = complaints.groupby('customer_id').size().reset_index(name='complaint_count')
customers = customers.merge(complaint_counts, on='customer_id', how='left')
customers['complaint_count'] = customers['complaint_count'].fillna(0).astype(int)
customers['is_repeat_complainer'] = (customers['complaint_count'] >= 2).astype(int)

# Vehicle risk score
vehicles['vehicle_risk_score'] = (
    (100 - vehicles['battery_health_pct']) * 0.4 +
    (vehicles['odometer_km'] / vehicles['odometer_km'].max()) * 100 * 0.3 +
    (vehicles['maintenance_status'] == 'InRepair').astype(int) * 30
)

print('Feature engineering complete.')
print(f'  Negative delivery times flagged: {deliveries["is_negative_time"].sum()}')
print(f'  Repeat complainers: {customers["is_repeat_complainer"].sum()}')
print(f'  Vehicle risk scores range: {vehicles["vehicle_risk_score"].min():.1f} - {vehicles["vehicle_risk_score"].max():.1f}')

In [ ]:

# STEP 4: CREATE MERGED ANALYTICAL DATASET


# Build comprehensive merged dataset
merged = (
    deliveries
    .merge(orders, on='order_id', how='left')
    .merge(drivers, on='driver_id', how='left')
    .merge(vehicles, on='vehicle_id', how='left')
    .merge(hubs, on='hub_id', how='left')
)

# Add incident counts per delivery
inc_counts = incidents.groupby('delivery_id').agg(
    incident_count=('incident_id', 'count'),
    has_vehicle_fault=('incident_type', lambda x: int('VehicleFault' in x.values)),
    has_battery_alert=('incident_type', lambda x: int('BatteryAlert' in x.values))
).reset_index()

merged = merged.merge(inc_counts, on='delivery_id', how='left')
merged['incident_count'] = merged['incident_count'].fillna(0).astype(int)
merged['has_vehicle_fault'] = merged['has_vehicle_fault'].fillna(0).astype(int)
merged['has_battery_alert'] = merged['has_battery_alert'].fillna(0).astype(int)

print(f'Merged analytical dataset: {merged.shape[0]} rows, {merged.shape[1]} columns')

In [ ]:

# ANALYSIS 1: Zone Performance Dashboard


fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1a: Failure rate by zone
zone_fail = merged.groupby('pickup_zone').agg(
    fail_rate=('is_failed', 'mean'),
    delay_rate=('is_delayed', 'mean'),
    total=('delivery_id', 'count')
).sort_values('fail_rate', ascending=False)

zone_fail[['fail_rate', 'delay_rate']].plot(kind='bar', ax=axes[0, 0], color=['#e74c3c', '#f39c12'])
axes[0, 0].set_title('Failure & Delay Rate by Zone', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Rate')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].legend(['Failure', 'Delay'])

# 1b: Cost distribution by zone
sns.boxplot(data=merged, x='pickup_zone', y='fuel_or_charge_cost',
            order=zone_fail.index, ax=axes[0, 1], palette='RdYlGn_r')
axes[0, 1].set_title('Fuel/Charge Cost Distribution by Zone', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# 1c: Route overrides by zone and status
override_data = merged.groupby(['pickup_zone', 'delivery_status'])['manual_route_override_count'].mean().unstack()
override_data.plot(kind='bar', ax=axes[1, 0], color=['#f39c12', '#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Avg Route Overrides by Zone & Status', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Avg Overrides')
axes[1, 0].tick_params(axis='x', rotation=45)

# 1d: Order value vs cost scatter
colours = {'OnTime': '#2ecc71', 'Delayed': '#f39c12', 'Failed': '#e74c3c'}
for status, colour in colours.items():
    subset = merged[merged['delivery_status'] == status]
    axes[1, 1].scatter(subset['order_value'], subset['fuel_or_charge_cost'],
                       c=colour, alpha=0.3, s=20, label=status)
axes[1, 1].set_title('Order Value vs Operational Cost', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Order Value (£)')
axes[1, 1].set_ylabel('Fuel/Charge Cost (£)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('zone_performance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key finding: Central zone has 19% failure rate (highest), while costs are')
print('relatively uniform across zones, meaning the financial drain comes from')
print('wasted effort on failed deliveries rather than cost structure differences.')

In [ ]:

# ANALYSIS 2: Temporal Patterns in Service Failures


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Failure rate by hour of day
hourly = merged.groupby('order_hour').agg(
    fail_rate=('is_failed', 'mean'),
    count=('delivery_id', 'count')
)

ax1 = axes[0]
ax1.bar(hourly.index, hourly['count'], alpha=0.3, color='#3498db', label='Order Volume')
ax1_twin = ax1.twinx()
ax1_twin.plot(hourly.index, hourly['fail_rate'] * 100, 'r-o', linewidth=2, label='Failure Rate %')
ax1.set_title('Order Volume & Failure Rate by Hour', fontsize=12, fontweight='bold')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Order Count')
ax1_twin.set_ylabel('Failure Rate (%)', color='red')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Failure rate by day of week
daily = merged.groupby('order_dayofweek').agg(
    fail_rate=('is_failed', 'mean'),
    delay_rate=('is_delayed', 'mean')
)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily.index = [day_names[i] for i in daily.index]

daily.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12'])
axes[1].set_title('Failure & Delay Rate by Day of Week', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Rate')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Failure', 'Delay'])

plt.tight_layout()
plt.savefig('temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print('Finding: Certain hours and days show elevated failure rates, suggesting')
print('staffing or resource allocation issues during specific time windows.')

In [ ]:

# ANALYSIS 3: Vehicle Fleet Risk Analysis



veh_perf = deliveries.groupby('vehicle_id').agg(
    total_deliveries=('delivery_id', 'count'),
    failed=('is_failed', 'sum'),
    fail_rate=('is_failed', 'mean'),
    avg_cost=('fuel_or_charge_cost', 'mean'),
    total_overrides=('manual_route_override_count', 'sum')
).reset_index()

veh_analysis = vehicles.merge(veh_perf, on='vehicle_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a: Battery health vs failure rate
scatter = axes[0].scatter(veh_analysis['battery_health_pct'], 
                          veh_analysis['fail_rate'] * 100,
                          c=veh_analysis['vehicle_risk_score'],
                          cmap='RdYlGn_r', s=60, alpha=0.7, edgecolors='grey')
plt.colorbar(scatter, ax=axes[0], label='Risk Score')
axes[0].set_title('Battery Health vs Failure Rate', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Battery Health (%)')
axes[0].set_ylabel('Failure Rate (%)')

# 3b: Odometer vs failure rate by maintenance status
for status in ['Active', 'InRepair', 'Scheduled']:
    subset = veh_analysis[veh_analysis['maintenance_status'] == status]
    axes[1].scatter(subset['odometer_km'], subset['fail_rate'] * 100,
                    label=status, alpha=0.6, s=50)
axes[1].set_title('Odometer vs Failure Rate by Maint. Status', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Odometer (km)')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].legend()

# 3c: Vehicle type distribution of maintenance status
maint_dist = vehicles.groupby(['vehicle_type', 'maintenance_status']).size().unstack(fill_value=0)
maint_dist.plot(kind='bar', stacked=True, ax=axes[2],
                color=['#2ecc71', '#e74c3c', '#f39c12'])
axes[2].set_title('Fleet Maintenance Status by Vehicle Type', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('vehicle_fleet_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Vehicles in repair: {(vehicles.maintenance_status=="InRepair").sum()} / {len(vehicles)} ({(vehicles.maintenance_status=="InRepair").mean()*100:.0f}%)')
print('30% of the fleet is either InRepair or Scheduled for maintenance.')
print('This significantly reduces available capacity and contributes to failures.')

In [ ]:

# ANALYSIS 4: Customer Complaint Journey Analysis


# Which customers are most at risk?
cust_risk = (
    customers
    .merge(
        orders.groupby('customer_id').agg(
            total_orders=('order_id', 'count'),
            avg_order_value=('order_value', 'mean')
        ).reset_index(), on='customer_id', how='left'
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 4a: Complaints by customer type
cust_type_comp = cust_risk.groupby('customer_type').agg(
    avg_complaints=('complaint_count', 'mean'),
    total_customers=('customer_id', 'count'),
    avg_loyalty=('loyalty_score', 'mean')
)

ax1 = axes[0]
bars = ax1.bar(cust_type_comp.index, cust_type_comp['avg_complaints'], color='#e74c3c', alpha=0.7)
ax1_twin = ax1.twinx()
ax1_twin.plot(cust_type_comp.index, cust_type_comp['avg_loyalty'], 'bo-', linewidth=2)
ax1.set_title('Avg Complaints & Loyalty by Customer Type', fontsize=11, fontweight='bold')
ax1.set_ylabel('Avg Complaints', color='red')
ax1_twin.set_ylabel('Avg Loyalty Score', color='blue')

# 4b: Complaint type by channel
comp_channel = complaints.groupby(['complaint_type', 'channel']).size().unstack(fill_value=0)
comp_channel.plot(kind='barh', stacked=True, ax=axes[1])
axes[1].set_title('Complaint Type by Reporting Channel', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('customer_complaint_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

# ANALYSIS 5: App Events - Digital Platform Performance


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 5a: API latency distribution by event type
event_order = app_events.groupby('event_type')['api_latency_ms'].mean().sort_values(ascending=False).index
sns.boxplot(data=app_events, x='event_type', y='api_latency_ms',
            order=event_order, ax=axes[0], palette='RdYlBu_r')
axes[0].set_title('API Latency by Event Type', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(y=500, color='red', linestyle='--', alpha=0.5, label='500ms threshold')
axes[0].legend()

# 5b: Success rate by event type
success = app_events.groupby('event_type')['success_flag'].agg(['mean', 'count'])
success = success.sort_values('mean')
bars = axes[1].barh(success.index, success['mean'] * 100,
                     color=['#e74c3c' if x < 0.9 else '#2ecc71' for x in success['mean']])
axes[1].set_title('Success Rate by Event Type', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Success Rate (%)')
axes[1].axvline(x=90, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('app_event_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key findings:')
print(f'  chat_escalated success rate: {app_events[app_events.event_type=="chat_escalated"]["success_flag"].mean()*100:.0f}% (critically low)')
print(f'  payment_retry success rate: {app_events[app_events.event_type=="payment_retry"]["success_flag"].mean()*100:.0f}% (revenue risk)')
print(f'  Avg latency for delivery_instruction_update: {app_events[app_events.event_type=="delivery_instruction_update"]["api_latency_ms"].mean():.0f}ms (highest)')

In [ ]:

# ANALYSIS 6: Cross-System Data Consistency Check


# Deliveries marked OnTime but with complaints
ontime_deliveries = deliveries[deliveries['delivery_status'] == 'OnTime']['order_id']
complaints_on_ontime = complaints[complaints['order_id'].isin(ontime_deliveries)]

# Deliveries marked OnTime but with incidents
ontime_delivery_ids = deliveries[deliveries['delivery_status'] == 'OnTime']['delivery_id']
incidents_on_ontime = incidents[incidents['delivery_id'].isin(ontime_delivery_ids)]

print('=== CROSS-SYSTEM DATA MISMATCH ANALYSIS ===')
print(f'Total OnTime deliveries: {len(ontime_deliveries)}')
print(f'Complaints filed against OnTime orders: {len(complaints_on_ontime)} ({len(complaints_on_ontime)/len(ontime_deliveries)*100:.1f}%)')
print(f'Incidents recorded for OnTime deliveries: {len(incidents_on_ontime)} ({len(incidents_on_ontime)/len(ontime_delivery_ids)*100:.1f}%)')

print(f'\nComplaint types on OnTime orders:')
print(complaints_on_ontime['complaint_type'].value_counts().to_string())

print(f'\nIncident types on OnTime deliveries:')
print(incidents_on_ontime['incident_type'].value_counts().to_string())

# Visualise the mismatch
fig, ax = plt.subplots(figsize=(8, 5))
status_issues = pd.DataFrame({
    'Status': ['OnTime', 'Delayed', 'Failed'],
    'With Complaints': [
        len(complaints[complaints['order_id'].isin(deliveries[deliveries['delivery_status']=='OnTime']['order_id'])]),
        len(complaints[complaints['order_id'].isin(deliveries[deliveries['delivery_status']=='Delayed']['order_id'])]),
        len(complaints[complaints['order_id'].isin(deliveries[deliveries['delivery_status']=='Failed']['order_id'])])
    ],
    'With Incidents': [
        len(incidents[incidents['delivery_id'].isin(deliveries[deliveries['delivery_status']=='OnTime']['delivery_id'])]),
        len(incidents[incidents['delivery_id'].isin(deliveries[deliveries['delivery_status']=='Delayed']['delivery_id'])]),
        len(incidents[incidents['delivery_id'].isin(deliveries[deliveries['delivery_status']=='Failed']['delivery_id'])])
    ]
})

x = np.arange(3)
width = 0.35
ax.bar(x - width/2, status_issues['With Complaints'], width, label='Complaints', color='#e74c3c')
ax.bar(x + width/2, status_issues['With Incidents'], width, label='Incidents', color='#3498db')
ax.set_xticks(x)
ax.set_xticklabels(status_issues['Status'])
ax.set_title('Complaints & Incidents by Delivery Status', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.savefig('cross_system_mismatch.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCRITICAL FINDING: 191 incidents and numerous complaints exist for orders')
print('marked OnTime. This is the cross-system mismatch the case study describes.')
print('The delivery system reports success while other systems record problems.')

In [ ]:

# ANALYSIS 7: Comprehensive Summary Statistics Table


summary = merged.groupby('hub_name').agg(
    deliveries=('delivery_id', 'count'),
    fail_rate=('is_failed', 'mean'),
    delay_rate=('is_delayed', 'mean'),
    avg_distance=('route_distance_km', 'mean'),
    avg_cost=('fuel_or_charge_cost', 'mean'),
    avg_overrides=('manual_route_override_count', 'mean'),
    avg_rating=('customer_rating_post_delivery', 'mean'),
    incidents=('incident_count', 'sum'),
    poc_missing=('proof_of_completion_missing', 'sum')
).round(3)

print('=== COMPREHENSIVE HUB PERFORMANCE SUMMARY ===')
print(summary.sort_values('fail_rate', ascending=False).to_string())

print('\n=== SERVICE TYPE PERFORMANCE ===')
svc_summary = merged.groupby('service_type').agg(
    deliveries=('delivery_id', 'count'),
    fail_rate=('is_failed', 'mean'),
    avg_value=('order_value', 'mean'),
    avg_cost=('fuel_or_charge_cost', 'mean'),
    margin=('order_value', lambda x: x.mean()),
).round(3)
print(svc_summary.sort_values('fail_rate', ascending=False).to_string())